Machine learning classifiers are obviously very valuable in areas such as diagnostics. In an ideal world we could simply feed someone's test results into a machine and it would reliably tell us what, if anything, they are sick with. The Wisconsin diagnostic breast cancer dataset (available: https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic ) is an interesting dataset where growths are classified into malignant ("M") or benign ("B") in column 1. We'll use this to test various machine learning models for their effectiveness at correctly classifying breast cancer, and use this to develop a triage system capable of getting expert-level performance on the task of classifying these samples.

The dataset uses "a digitized image of a fine needle aspirate (FNA) of a breast mass". We'll speculate that a human specialist has an accuracy (% of images classified correctly), sensitivity (% of malignant images correctly identified as such), and specificity (% of benign samples correctly identified as such) of 95%, and a false positive rate (% of benign images incorrectly labelled as malignant) and a false negative rate (% of malignant samples incorrectly labelled as benign) of 5%. Therefore our ideal algorithm would have acc, spec, and sens > 95% and FPR & FNR < 5%. It's also worth noting that while false positives still matter, it's generally better to incorrectly identify someone as having cancer rather than to incorrectly identify them as not having cancer. Unnecessary medical treatments are still bad but they're less bad than death.

This code is of course not a substitute for proper medical advice, and there are a number of issues with applying machine learning to real world contexts as sensitive as oncology. Neither machine learning systems nor human medical practitioners are perfect and while this code is a toy example, proper real world systems should use both humans and computers appropriately. Good use of ML augments human capabilities but does not replace them.

Imports & Seeding

In [1]:
import random as r
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import confusion_matrix, accuracy_score

Loading data

In [2]:
cancer_data = pd.read_csv("wdbc.data", header = None)

with open("wdbc.names", "r") as namesFile:
    names_content = namesFile.read()

Displaying explanation and part of the data

In [3]:
print(names_content)
print(cancer_data.head())

1. Title: Wisconsin Diagnostic Breast Cancer (WDBC)

2. Source Information

a) Creators: 

	Dr. William H. Wolberg, General Surgery Dept., University of
	Wisconsin,  Clinical Sciences Center, Madison, WI 53792
	wolberg@eagle.surgery.wisc.edu

	W. Nick Street, Computer Sciences Dept., University of
	Wisconsin, 1210 West Dayton St., Madison, WI 53706
	street@cs.wisc.edu  608-262-6619

	Olvi L. Mangasarian, Computer Sciences Dept., University of
	Wisconsin, 1210 West Dayton St., Madison, WI 53706
	olvi@cs.wisc.edu 

b) Donor: Nick Street

c) Date: November 1995

3. Past Usage:

first usage:

	W.N. Street, W.H. Wolberg and O.L. Mangasarian 
	Nuclear feature extraction for breast tumor diagnosis.
	IS&T/SPIE 1993 International Symposium on Electronic Imaging: Science
	and Technology, volume 1905, pages 861-870, San Jose, CA, 1993.

OR literature:

	O.L. Mangasarian, W.N. Street and W.H. Wolberg. 
	Breast cancer diagnosis and prognosis via linear programming. 
	Operations Research, 43(4), pag

In [4]:
#Shuffling the data
cancer_data = cancer_data.sample(frac = 1, random_state = 42).reset_index(drop = True)

#Sorting into measurements and labels
cancer_measurements = cancer_data.iloc[:, 2:]
cancer_labels = cancer_data.iloc[:, 1]
print(cancer_measurements.head())
print(cancer_labels.head())

#Split into training, validation, and testing
train_threshold = int(0.5 * len(cancer_measurements))
val_threshold = int(0.75 * len(cancer_measurements))
training_data = cancer_measurements[:train_threshold]
validation_data = cancer_measurements[train_threshold:val_threshold]
testing_data = cancer_measurements[val_threshold:]
training_labels = cancer_labels[:train_threshold]
validation_labels = cancer_labels[train_threshold:val_threshold]
testing_labels = cancer_labels[val_threshold:]

#Standardize data and map onto integers
scaler = StandardScaler()
training_scaled = scaler.fit_transform(training_data)
validation_scaled = scaler.transform(validation_data)
testing_scaled = scaler.transform(testing_data)
training_y = training_labels.map({'M': 1, 'B': 0}).values
validation_y = validation_labels.map({'M': 1, 'B': 0}).values
testing_y = testing_labels.map({'M': 1, 'B': 0}).values

      2      3       4       5        6       7        8        9       10  \
0  12.47  18.60   81.09   481.9  0.09965  0.1058  0.08005  0.03821  0.1925   
1  18.94  21.31  123.60  1130.0  0.09009  0.1029  0.10800  0.07951  0.1582   
2  15.46  19.48  101.70   748.9  0.10920  0.1223  0.14660  0.08087  0.1931   
3  12.40  17.68   81.47   467.8  0.10540  0.1316  0.07741  0.02799  0.1811   
4  11.54  14.44   74.65   402.9  0.09984  0.1120  0.06737  0.02594  0.1818   

        11  ...     22     23      24      25      26      27      28  \
0  0.06373  ...  14.97  24.64   96.05   677.9  0.1426  0.2378  0.2671   
1  0.05461  ...  24.86  26.58  165.90  1866.0  0.1193  0.2336  0.2687   
2  0.05796  ...  19.26  26.00  124.90  1156.0  0.1546  0.2394  0.3791   
3  0.07102  ...  12.88  22.91   89.61   515.8  0.1450  0.2629  0.2403   
4  0.06782  ...  12.26  19.68   78.78   457.8  0.1345  0.2118  0.1797   

        29      30       31  
0  0.10150  0.3014  0.08750  
1  0.17890  0.2551  0.06589  
2 

# K-Means

K-Means is unsupervised and ignores the labels, but we can use it to sort the data into two clusters. We'll see whether these clusters line up nicely with the diagnoses or not.

In [5]:
#Create, fit, and map onto labels
kmeans = KMeans(n_clusters = 2, random_state = 1, n_init = 10)
kmeans.fit(training_scaled)
predicted_clusters = kmeans.predict(testing_scaled)

#How do the predicted clusters compare to the diagnostic labels?
print("\nCluster assignments:", predicted_clusters[:10])
print("Actual labels:", list(testing_labels[:10]))
true_labels = testing_labels.map({'M': 1, 'B': 0}).values

#Confusion Matrix
confusion = confusion_matrix(true_labels, predicted_clusters)
print("\nConfusion Matrix:")
print(confusion)

#Accuracy
acc = max(
    accuracy_score(true_labels, predicted_clusters),
    accuracy_score(true_labels, 1 - predicted_clusters)
)
print(f"\nCluster-label alignment accuracy: {100 * acc:.1f}%")


Cluster assignments: [0 1 0 0 1 1 0 1 0 0]
Actual labels: ['M', 'B', 'M', 'M', 'B', 'B', 'M', 'B', 'M', 'M']

Confusion Matrix:
[[ 1 81]
 [48 13]]

Cluster-label alignment accuracy: 90.2%


So in this case K-Means is calling malignant "0" and benign "1". That's fine, the K-Means labels are arbitrary but we can still use them to classify 103 / 114 of our test cases correctly for a success rate of 90.4%, that's not bad for a very simple approach!

# Gaussian Mixture Model

GMM predicts the probability of belonging to each cluster rather than just cluster memberships, which can be valuable as we can see not just the memberships but how sure our model is. If the probability is not near 0 or near 1, maybe it's time to pass it to a human!

In [6]:
#Create, fit, predict
gmm = GaussianMixture(n_components = 2, random_state = 1)
gmm.fit(training_scaled)
predicted_clusters = gmm.predict(testing_scaled)

#Confusion Matrix
true_labels = testing_labels.map({'M': 1, 'B': 0}).values
confusion = confusion_matrix(true_labels, predicted_clusters)
print("\nConfusion Matrix:")
print(confusion)

#Accuracy
acc = max(
    accuracy_score(true_labels, predicted_clusters),
    accuracy_score(true_labels, 1 - predicted_clusters)
)
print(f"\nCluster-label alignment accuracy: {100 * acc:.1f}%")


Confusion Matrix:
[[79  3]
 [ 6 55]]

Cluster-label alignment accuracy: 93.7%


# Artificial Neural Network

Does an MLP classifier do better?

In [7]:
#Create, train, predict
mlp = MLPClassifier(
    hidden_layer_sizes = (16, 8),  
    activation = 'relu',            
    solver = 'adam',                
    max_iter = 500               
)
mlp.fit(training_scaled, training_y)
predictions = mlp.predict(testing_scaled)

#Confusion matrix
confusion = confusion_matrix(testing_y, predictions)
print("\nConfusion Matrix:")
print(confusion)

#Metrics
tn, fp, fn, tp = confusion.ravel()
accuracy = (tp + tn) / (tp + tn + fp + fn)
sensitivity = tp / (tp + fn)         
specificity = tn / (tn + fp)        
fpr = fp / (fp + tn)                  
fnr = fn / (fn + tp)   
acc = accuracy_score(testing_y, predictions)
print(f"\nNeural Network Accuracy: {100 * acc:.1f}%")
print(f"Sensitivity (TPR): {100 * sensitivity:.1f}%")
print(f"Specificity (TNR): {100 * specificity:.1f}%")
print(f"False Positive Rate (FPR): {100 * fpr:.1f}%")
print(f"False Negative Rate (FNR): {100 * fnr:.1f}%")


Confusion Matrix:
[[81  1]
 [ 5 56]]

Neural Network Accuracy: 95.8%
Sensitivity (TPR): 91.8%
Specificity (TNR): 98.8%
False Positive Rate (FPR): 1.2%
False Negative Rate (FNR): 8.2%


# Majority-Voting Ensemble

All of our methods are getting 90%+ accuracy. If they fail on different samples, we might be able to get better performance by having all three models predict and going with whatever is the most common prediction! This also lets us flag to a human, since if the models disagree then that's an indicator that we might be unsure.

In [8]:
#Inverting because K-means maps M -> 0 and B -> 1
def predictKM(data):
    return(1 - kmeans.predict(data))

def predictGMM(data):
    return(gmm.predict(data))

class Majority_Voting_Ensemble:
    def __init__(self, methods):
        self.methods = methods
    def flag(self, data, predictions):
        print(f"Data point flagged for uncertainty {data} refer to specialist")
        print(f"predictions: {predictions}")
    def predictSample(self, data, silencing = False):
        predictions = []
        for method in self.methods:
            predictions.append(method(data))
        #Flag unsure cases if not silenced
        if np.mean(predictions) != 0 and np.mean(predictions) != 1 and not silencing:
            self.flag(data, predictions)
        return([0 if np.mean(predictions) < 0.5 else 1])
    def predict(self, data, silencing = False):
        return([self.predictSample([sample], silencing = silencing) for sample in data])

#Create, predict
majority_ensemble = Majority_Voting_Ensemble([predictGMM, predictKM, mlp.predict])
predictions = majority_ensemble.predict(testing_scaled, silencing = True)

#Confusion matrix
confusion = confusion_matrix(testing_y, predictions)
print("\nConfusion Matrix:")
print(confusion)

#Metrics
tn, fp, fn, tp = confusion.ravel()
accuracy = (tp + tn) / (tp + tn + fp + fn)
sensitivity = tp / (tp + fn)         
specificity = tn / (tn + fp)        
fpr = fp / (fp + tn)                  
fnr = fn / (fn + tp)                 
print(f"\nMajority Voting Ensemble Accuracy: {100 * accuracy:.1f}%")
print(f"Sensitivity (TPR): {100 * sensitivity:.1f}%")
print(f"Specificity (TNR): {100 * specificity:.1f}%")
print(f"False Positive Rate (FPR): {100 * fpr:.1f}%")
print(f"False Negative Rate (FNR): {100 * fnr:.1f}%")


Confusion Matrix:
[[81  1]
 [ 6 55]]

Majority Voting Ensemble Accuracy: 95.1%
Sensitivity (TPR): 90.2%
Specificity (TNR): 98.8%
False Positive Rate (FPR): 1.2%
False Negative Rate (FNR): 9.8%


# Cautious Ensemble

So our majority-voting ensemble didn't do any better than the neural network alone, but at least now we cal flag uncertain cases by setting silencing = False. In a medical context false negatives matter a lot more than false positives, so we can make our ensemble cautious by forcing it to predict malignant if any of the models predict malignant.

In [9]:
class Cautious_Ensemble:
    def __init__(self, methods):
        self.methods = methods
    def flag(self, data, predictions):
        print(f"Data point flagged for uncertainty {data} refer to specialist")
        print(f"predictions: {predictions}")
    def predictSample(self, data, silencing = True):
        predictions = []
        for method in self.methods:
            predictions.append(method(data))
        #Flag unsure cases if not silenced
        if np.mean(predictions) != 0 and np.mean(predictions) != 1 and not silencing:
            self.flag(data, predictions)
        return([0 if np.mean(predictions) == 0 else 1])
    def predict(self, data, silencing = True):
        return([self.predictSample([sample], silencing = silencing) for sample in data])

#Create, predict
cautious_ensemble = Cautious_Ensemble([predictGMM, predictKM, mlp.predict])
predictions = cautious_ensemble.predict(testing_scaled, silencing = True)

#Confusion matrix
confusion = confusion_matrix(testing_y, predictions)
print("\nConfusion Matrix:")
print(confusion)

#Metrics
tn, fp, fn, tp = confusion.ravel()
accuracy = (tp + tn) / (tp + tn + fp + fn)
sensitivity = tp / (tp + fn)         
specificity = tn / (tn + fp)        
fpr = fp / (fp + tn)                  
fnr = fn / (fn + tp)                 
print(f"\nCautious Ensemble Accuracy: {100 * accuracy:.1f}%")
print(f"Sensitivity (TPR): {100 * sensitivity:.1f}%")
print(f"Specificity (TNR): {100 * specificity:.1f}%")
print(f"False Positive Rate (FPR): {100 * fpr:.1f}%")
print(f"False Negative Rate (FNR): {100 * fnr:.1f}%")


Confusion Matrix:
[[78  4]
 [ 3 58]]

Cautious Ensemble Accuracy: 95.1%
Sensitivity (TPR): 95.1%
Specificity (TNR): 95.1%
False Positive Rate (FPR): 4.9%
False Negative Rate (FNR): 4.9%


# Bigger model

Our ensembles are "almost as good as the specialist" in that we have acc, TPR, and TNR ~ 93% and FPR/FNR ~ 6%, when our targets were 95% and 5%. There is still some value in getting a less accurate response more quickly as the waiting times for seeing a specialist can be long, but let's see if we can do better. Our first MLP classiffier was relatively small, so let's see how a bigger model can do!

In [10]:
#Create, train, predict
bigger_mlp = MLPClassifier(
    hidden_layer_sizes = (128, 64, 32, 16),
    activation = 'relu',
    solver = 'adam',
    learning_rate_init = 0.001,
    max_iter = 1000
)

bigger_mlp.fit(training_scaled, training_y)
predictions = bigger_mlp.predict(testing_scaled)

#Confusion matrix
confusion = confusion_matrix(testing_y, predictions)
print("\nConfusion Matrix:")
print(confusion)

#Metrics
tn, fp, fn, tp = confusion.ravel()
accuracy = (tp + tn) / (tp + tn + fp + fn)
sensitivity = tp / (tp + fn)         
specificity = tn / (tn + fp)        
fpr = fp / (fp + tn)                  
fnr = fn / (fn + tp)   
acc = accuracy_score(testing_y, predictions)
print(f"\nBigger Neural Network Accuracy: {100 * acc:.1f}%")
print(f"Sensitivity (TPR): {100 * sensitivity:.1f}%")
print(f"Specificity (TNR): {100 * specificity:.1f}%")
print(f"False Positive Rate (FPR): {100 * fpr:.1f}%")
print(f"False Negative Rate (FNR): {100 * fnr:.1f}%")


Confusion Matrix:
[[81  1]
 [ 6 55]]

Bigger Neural Network Accuracy: 95.1%
Sensitivity (TPR): 90.2%
Specificity (TNR): 98.8%
False Positive Rate (FPR): 1.2%
False Negative Rate (FNR): 9.8%


# Cautious Ensemble 2.0

The bigger MLP did well. Let's see what happens if we train several big MLPs and have them act as the deciders for a cautious ensemble.

In [11]:
big1 = MLPClassifier(
    hidden_layer_sizes = (128, 64, 32, 16),
    activation = 'relu',
    solver = 'adam',
    learning_rate_init = 0.001,
    max_iter = 1000
)
big1.fit(training_scaled, training_y)

big2 = MLPClassifier(
    hidden_layer_sizes = (128, 64, 32, 16),
    activation = 'relu',
    solver = 'adam',
    learning_rate_init = 0.001,
    max_iter = 1000
)
big2.fit(training_scaled, training_y)

big3 = MLPClassifier(
    hidden_layer_sizes = (128, 64, 32, 16),
    activation = 'relu',
    solver = 'adam',
    learning_rate_init = 0.001,
    max_iter = 1000
)
big3.fit(training_scaled, training_y)

#Create, predict
cautious_ensemble2 = Cautious_Ensemble([predictKM, predictGMM, big1.predict, big2.predict, big3.predict])
predictions = cautious_ensemble2.predict(testing_scaled, silencing = True)

#Confusion matrix
confusion = confusion_matrix(testing_y, predictions)
print("\nConfusion Matrix:")
print(confusion)

#Metrics
tn, fp, fn, tp = confusion.ravel()
accuracy = (tp + tn) / (tp + tn + fp + fn)
sensitivity = tp / (tp + fn)         
specificity = tn / (tn + fp)        
fpr = fp / (fp + tn)                  
fnr = fn / (fn + tp)                 
print(f"\nCautious Ensemble Accuracy: {100 * accuracy:.1f}%")
print(f"Sensitivity (TPR): {100 * sensitivity:.1f}%")
print(f"Specificity (TNR): {100 * specificity:.1f}%")
print(f"False Positive Rate (FPR): {100 * fpr:.1f}%")
print(f"False Negative Rate (FNR): {100 * fnr:.1f}%")


Confusion Matrix:
[[77  5]
 [ 2 59]]

Cautious Ensemble Accuracy: 95.1%
Sensitivity (TPR): 96.7%
Specificity (TNR): 93.9%
False Positive Rate (FPR): 6.1%
False Negative Rate (FNR): 3.3%


# Triage

So our best models are the Bigger MLP and the Cautious Ensemble 2.0. If we get a new sample our "best guess" is the prediction from the bigger MLP, but we'll also predict them with the CE2 and we send anyone who CE2 thinks might have cancer to a specialist, but also tell them we don't think they have cancer but we want to be sure. That way, we are less likely to worry people unnecessarily but also less likely to miss cancer.

In [12]:
class Triage:
    def __init__(self, predictor, referrer):
        self.predictor = predictor
        self.referrer = referrer

    def predictSample(self, sample):
        return(self.predictor.predict(sample))

    def predict(self, data):
        return([self.predictSample([sample]) for sample in data])

    def referSample(self, sample):
        return(self.referrer.predict(sample))

    def refer(self, data):
        return([self.referSample([sample]) for sample in data])

triage = Triage(bigger_mlp, cautious_ensemble2)
predictions = triage.predict(validation_scaled)
referrals = triage.refer(validation_scaled)


true_cancer = np.array(validation_y).astype(int).flatten()
told_cancer = np.array(predictions).astype(int).flatten()
referred = np.array(referrals).astype(int).flatten()

triage_df = pd.DataFrame({
    'TrueCancer': true_cancer,
    'ToldCancer': told_cancer,
    'Referred': referred
})

summary = (
    triage_df
    .groupby(['TrueCancer', 'ToldCancer', 'Referred'], as_index = False)
    .size()
    .rename(columns = {'size': 'Count'})
    .sort_values(['TrueCancer', 'ToldCancer', 'Referred'])
    .reset_index(drop = True)
)
print(summary)

   TrueCancer  ToldCancer  Referred  Count
0           0           0         0     80
1           0           0         1      8
2           0           1         1      1
3           1           0         0      1
4           1           0         1      5
5           1           1         1     47


So this describes all the possible outcomes and how often they occur in validation:

- 80 people did not have cancer and were told they do not have cancer and were not referred for further treatment.
- 8 people did not have cancer and were told they probably did not have cancer but were referred to a specialist just in case.
- 1 person who did not have cancer was incorrectly told they had cancer and referred to a specialist.
- 1 person did have cancer but was incorrectly told they do not and not referred for further treatment.
- 4 people had cancer and were told they probably don't have cancer but were referred to a specialist just in case.
- 48 people had cancer, were told they had cancer, and were sent to the specialist for treatment.

So this is quite a good result! There was 1 major problem (the person who was told they do not have cancer and not referred to the specialist but who did have cancer), 4 nasty surprises that were properly managed (they did have cancer but were told they don't, but were sent to the specialist just in case), 8 inconveniences (we don't think you have cancer but just in case we'll refer you anyway), and 1 unnececessary scare (someone told they have cancer when they don't). For 128 people being quickly diagnosed and correctly triaged, that's not bad. 

Of course one person dying because they were told they do not have cancer when they do is still one too many. For our speculated FNR we would expect the human expert to incorrectly classify 2-3 people in this way rather than 1 so from a certain perspective that's 2 lives saved, but that wouldn't be much consolation if you are the loved one of someone who was incorrectly classified and who died as a result. 

Therefore the correct use of a system such as this is not to actually accept or reject patients but to decide whether or not to fast track them. A system which puts the 50 patients who were told they have cancer on a Tier 3 fast track, the 11 who were told they do not have cancer but referred them to a specialist on a Tier 2 fast track, and everyone else on a Tier 1 track seems like an acceptable way to speed up the diagnostic process while still ensuring everyone gets seen by the specialist eventually.